
# Desafio 1 — Exploração Inicial da Base

## Objetivo
Conhecer a estrutura da base de dados do formulário sobre o uso de IA no ICH/UFJF, identificando suas dimensões, tipos de dados e valores ausentes.

## Análise
Utilizei a biblioteca Pandas e os seguintes métodos:

- `pd.read_csv()` — carreguei o arquivo CSV.
- `df.head()` — visualizei as primeiras respostas.
- `df.shape` — identifiquei a dimensão da base.
- `df.info()` — verifiquei os tipos de dados.
- `df.isnull().sum()` — contei os valores ausentes.

## Resultados
- **72 respostas e 18 colunas.**
- Todas as colunas foram inicialmente interpretadas como texto (`str`).
- Identifiquei valores ausentes em algumas perguntas, principalmente nas destinadas a perfis específicos e nas questões abertas.

## Conclusão
Consegui compreender a estrutura inicial da base e identificar pontos que precisam de investigação antes do tratamento dos dados.


In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('../data/ia.csv')

In [ ]:
df.info()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()


# Desafio 2 — Perfil dos Respondentes

## Objetivo
Identificar os perfis dos participantes e investigar a relação entre o enquadramento e os valores ausentes.

## Análise
Utilizei os métodos:

- `unique()` — identifiquei as categorias de enquadramento.
- `value_counts()` — contei os participantes de cada categoria.
- `isnull()` — investiguei os valores ausentes.

## Resultados

| Enquadramento             | Quantidade |
| Docente                   | 68         |
| Técnico-Administrativo    | 3          |
| Discente de Pós-Graduação | 1          |
| **Total**                 | **72**     |

Identifiquei:
- **71 valores ausentes** na pergunta destinada aos discentes.
- **4 valores ausentes** na pergunta destinada aos docentes.

Essas quantidades correspondem aos participantes que não pertencem aos respectivos perfis.

## Conclusão
Identifiquei que os docentes representam 94,4% dos participantes, os Técnico-Administrativo representam 4.17% e o Discente de Pós-Graduação (Mestrado/Doutorado) representa 1.39%. Também observei que os valores ausentes nessas perguntas são compatíveis com a lógica do formulário, indicando possíveis ausências estruturais, ainda a confirmar por filtragem.


In [ ]:
df['Enquadramento'].unique()

In [ ]:
df['Enquadramento'].value_counts()

In [ ]:
(df['Enquadramento'].value_counts(normalize=True) * 100).map('{:.2f}%'.format)


# Desafio 3 — Identificação de Duplicatas e Registros Suspeitos

## Objetivo
Investigar possíveis duplicatas e respostas de teste na base de dados do formulário do ICH/UFJF.

## Análise
Utilizei os seguintes métodos:

- `duplicated()` — identifiquei e-mails repetidos e duplicatas completas.
- `iloc[]` — comparei duas submissões do mesmo participante.
- `str.contains()` — procurei respostas contendo "Teste".
- `apply()` e `any(axis=1)` — ampliei a busca para todas as colunas.
- `sum()` — contei os registros encontrados.

## Resultados
- Identifiquei 2 submissões vinculadas ao mesmo e-mail.
- Não encontrei linhas completamente duplicadas.
- Localizei 1 registro contendo "Teste", no índice 1.
- Nesse registro, encontrei 6 campos preenchidos exatamente com "Teste".

## Conclusão
Identifiquei uma provável submissão de teste no índice 1, que também está associada a um e-mail repetido.

Considerei esse registro suspeito e recomendo validar sua exclusão com o professor antes de realizar a limpeza definitiva. Mantive a base original sem alterações.


In [ ]:
df['Endereço de e-mail'].duplicated(keep=False).sum()

In [ ]:
df.duplicated().sum()

In [ ]:

quantidade_testes = (
    df.astype('string')
    .apply(
        lambda coluna: coluna.str.strip().str.casefold().eq('teste')
    )
    .fillna(False)
    .sum(axis=1)
)

print("Registros suspeitos:", (quantidade_testes >= 5).sum())
print("Maior quantidade de campos com Teste:", quantidade_testes.max())


# DESAFIO 4

In [ ]:
import json

with open('data/mapeamentos_categorias.json', 'r', encoding='utf-8') as arquivo:
    mapeamentos = json.load(arquivo)

mapeamento_docente = mapeamentos['orientacao_docente']
mapeamento_praticas = mapeamentos['praticas_ensino']
mapeamento_sigilo = mapeamentos['sigilo_documentos']
mapeamento_portaria = mapeamentos['portaria_cnpq']
mapeamento_opcoes = mapeamentos['praticas_opcoes_mencionadas']
mapeamento_outros_pesquisa = mapeamentos['outros_uso_pesquisa']
mapeamento_outros_riscos = mapeamentos['outros_riscos_gestao']

In [ ]:

df_limpo = df.copy()

quantidade_teste = (
    df_limpo.astype('string')
    .apply(
        lambda coluna: coluna.str.strip().str.casefold().eq('teste')
    )
    .fillna(False)
    .sum(axis=1)
)

registros_teste = quantidade_teste >= 5

print("Registros de teste identificados:", registros_teste.sum())

assert registros_teste.sum() == 1, (
    "Quantidade inesperada de registros de teste. Revisar antes de excluir."
)

df_limpo = df_limpo.loc[~registros_teste].copy()

print("Registros restantes:", len(df_limpo))


In [ ]:
df_limpo.shape

In [ ]:
df_limpo = df_limpo.reset_index(drop=True)

In [ ]:
df_limpo.index

In [ ]:
df_limpo.columns

In [ ]:
df_limpo.columns = ['data_resposta', 'email', 'fe_enquadramento', 'fe_orientacao_docente','fe_orientacao_discente', 'fe_departamento', 
                    'multi_uso_ia_pesquisa', 'fe_opiniao_portaria_cnpq', 'ab_riscos_pesquisa', 'ab_beneficios_pesquisa', 'fe_familiaridade_ia', 
                    'fe_praticas_ensino_ia', 'ab_preparacao_uso_etico', 'multi_riscos_gestao', 'fe_sigilo_documentos', 'ab_diretrizes_transparencia', 
                    'ab_comentarios_politica', 'ab_outros_topicos']

In [ ]:
df_limpo.columns

In [ ]:
df.columns

In [ ]:
df_limpo.info()

In [ ]:
df_limpo['data_resposta'] = pd.to_datetime(
    df_limpo['data_resposta'],
    format='%d/%m/%Y %H:%M:%S'
)

In [ ]:
df_limpo['data_resposta'].dtype

In [ ]:
df_limpo.info()

In [ ]:
df_limpo['fe_enquadramento'].unique()

In [ ]:
df_limpo['fe_enquadramento'].value_counts(dropna=False)

In [ ]:
df_limpo['fe_familiaridade_ia'].unique()

In [ ]:
df_limpo['fe_familiaridade_ia'].value_counts(dropna=False)

In [ ]:
(df_limpo['fe_enquadramento'] != df_limpo['fe_enquadramento'].str.strip()).sum()

In [ ]:
(df_limpo['fe_familiaridade_ia'] != df_limpo['fe_familiaridade_ia'].str.strip()).sum()

In [ ]:
colunas_fechadas = [
    'fe_enquadramento',
    'fe_orientacao_docente',
    'fe_orientacao_discente',
    'fe_familiaridade_ia',
    'fe_praticas_ensino_ia',
    'fe_sigilo_documentos'
]

for coluna in colunas_fechadas:
    print('Coluna:', coluna)
    print('Categorias distintas:', df_limpo[coluna].nunique())
    print('Valores ausentes:', df_limpo[coluna].isnull().sum())
    print('--------------------')

In [ ]:
df_limpo['fe_familiaridade_ia'].value_counts(dropna=False).head(10)

In [ ]:
opcoes_pesquisa = {
    'levantamento_bibliografico': 'Levantamento bibliográfico',
    'revisao_bibliografica': 'Revisão bibliográfica',
    'sintese_textos': 'Síntese de textos',
    'analise_dados': 'Análise de dados (qualitativos ou quantitativos)',
    'traducao_textos': 'Tradução de textos',
    'revisao_textos': 'Revisão de textos',
    'geracao_ideias': 'Geração de ideias ou estruturação de argumentos',
    'redacao_parcial': 'Redação parcial de seções (ex: introdução, metodologia, conclusão...)',
    'redacao_total': 'Redação total de seções (ex: introdução, metodologia, conclusão...)',
    'transcricao': 'Transcrição',
    'nunca_utilizei': 'Nunca utilizei',
    'nao_se_aplica': 'Não se aplica a mim'
}


In [ ]:

df_uso = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_pesquisa.items():
    df_uso[nome] = (
        df_limpo['multi_uso_ia_pesquisa']
        .str.contains(alternativa, regex=False, na=False)
    )



In [ ]:

frequencia_uso = df_uso.sum().sort_values(ascending=False)

frequencia_uso


In [ ]:
import re

padrao_pesquisa = '|'.join(
    re.escape(opcao)
    for opcao in sorted(
        opcoes_pesquisa.values(),
        key=len,
        reverse=True
    )
)
restos_pesquisa = (
    df_limpo['multi_uso_ia_pesquisa']
    .fillna('')
    .str.replace(padrao_pesquisa, '', regex=True)
    .str.strip(' ,;')
)
outros_pesquisa = df_limpo.loc[
    restos_pesquisa.ne(''),
    ['multi_uso_ia_pesquisa']
].copy()
outros_pesquisa['texto_outro'] = restos_pesquisa[
    restos_pesquisa.ne('')
]
print('Respostas personalizadas:', len(outros_pesquisa))


In [ ]:
opcoes_riscos = {
    'violacoes_sigilo': 'Violações de sigilo',
    'informacoes_enviesadas': 'Informações enviesadas',
    'precarizacao_trabalho': 'Precarização do trabalho humano',
    'sem_riscos': 'Não vejo riscos significativos'
}

import re

df_riscos = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_riscos.items():
    padrao = rf'(?:^|,\s*){re.escape(alternativa)}(?=,|$)'

    df_riscos[nome] = (
        df_limpo['multi_riscos_gestao']
        .str.contains(padrao, regex=True, na=False)
    )



In [ ]:
frequencia_riscos = (
    df_riscos.sum()
    .sort_values(ascending=False)
)

frequencia_riscos

In [ ]:
padrao_riscos = '|'.join(
    re.escape(opcao)
    for opcao in sorted(
        opcoes_riscos.values(),
        key=len,
        reverse=True
    )
)

restos_riscos = (
    df_limpo['multi_riscos_gestao']
    .fillna('')
    .str.replace(padrao_riscos, '', regex=True)
    .str.strip(' ,;')
)

outros_riscos = df_limpo.loc[
    restos_riscos.ne(''),
    ['multi_riscos_gestao']
].copy()

outros_riscos['texto_outro'] = restos_riscos[
    restos_riscos.ne('')
]

print('Respostas personalizadas:', len(outros_riscos))


In [ ]:
df_limpo['portaria_cnpq_categoria'] = (
    df_limpo['fe_opiniao_portaria_cnpq']
    .replace(mapeamento_portaria)
)

df_limpo['sigilo_categoria'] = (
    df_limpo['fe_sigilo_documentos']
    .replace(mapeamento_sigilo)
)

df_limpo['orientacao_docente_categoria'] = (
    df_limpo['fe_orientacao_docente']
    .replace(mapeamento_docente)
)

df_limpo['praticas_ensino_categoria'] = (
    df_limpo['fe_praticas_ensino_ia']
    .str.strip()
    .replace(mapeamento_praticas)
)

df_limpo['praticas_ensino_opcoes_mencionadas'] = (
    df_limpo['fe_praticas_ensino_ia']
    .map(mapeamento_opcoes)
    .astype('string')
)

df_limpo['uso_ia_outro_categoria'] = (
    restos_pesquisa.map(mapeamento_outros_pesquisa)
)

df_limpo['riscos_outro_categoria'] = (
    restos_riscos.map(mapeamento_outros_riscos)
)

print('Os 7 mapeamentos foram aplicados!')
print('Registros:', len(df_limpo))

In [ ]:
esperado = {
    'portaria_cnpq_categoria': 68,
    'sigilo_categoria': 71,
    'orientacao_docente_categoria': 67,
    'praticas_ensino_categoria': 71,
    'praticas_ensino_opcoes_mencionadas': 7,
    'uso_ia_outro_categoria': 8,
    'riscos_outro_categoria': 5
}

assert len(df_limpo) == 71

for coluna, quantidade in esperado.items():
    preenchidos = df_limpo[coluna].notna().sum()
    assert preenchidos == quantidade, f'Erro em {coluna}'
    print(f'{coluna}: {preenchidos} preenchidos')

assert df_limpo.index.equals(df_uso.index)
assert df_limpo.index.equals(df_riscos.index)

print('\nTodas as validações passaram!')

In [ ]:
nao_usa = (
    df_uso['nunca_utilizei'] |
    df_uso['nao_se_aplica']
)

usos_positivos = df_uso.drop(
    columns=['nunca_utilizei', 'nao_se_aplica']
).any(axis=1)

conflitos_uso = nao_usa & usos_positivos

print(
    "Possíveis contradições no uso de IA:",
    conflitos_uso.sum()
)


In [ ]:
sem_riscos = df_riscos['sem_riscos']

riscos_identificados = df_riscos.drop(
    columns=['sem_riscos']
).any(axis=1)

conflitos_riscos = sem_riscos & riscos_identificados

print(
    "Possíveis contradições nos riscos:",
    conflitos_riscos.sum()
)


In [ ]:

print("Registros originais limpos:", len(df_limpo))
print("Registros dos indicadores de uso:", len(df_uso))
print("Registros dos indicadores de riscos:", len(df_riscos))

print(
    "Índices de uso correspondem:",
    df_limpo.index.equals(df_uso.index)
)

print(
    "Índices de riscos correspondem:",
    df_limpo.index.equals(df_riscos.index)
)


In [ ]:
df_uso_validado = pd.DataFrame(index=df_limpo.index)

for nome, alternativa in opcoes_pesquisa.items():

    padrao = rf'(?:^|,\s*){re.escape(alternativa)}(?=,\s*|$)'

    df_uso_validado[nome] = (
        df_limpo['multi_uso_ia_pesquisa']
        .str.contains(padrao, regex=True, na=False)
    )

divergencias_uso = df_uso != df_uso_validado

print(
    'Indicadores diferentes:',
    divergencias_uso.to_numpy().sum()
)

print(
    'Participantes com alguma diferença:',
    divergencias_uso.any(axis=1).sum()
)

print(
    'Índices para revisão:',
    df_limpo.index[divergencias_uso.any(axis=1)].tolist()
)
